# AIHub T자형 교차로 데이터 점검 02 — 밝기 중복 후보 찾기

이번 노트북은 영상을 삭제하거나 대표 영상을 확정하지 않습니다. 각 영상의 같은 위치에서 5개 프레임을 읽고, 전체 밝기 변화에 둔감한 pHash를 만든 뒤 **가장 구조가 비슷한 영상 쌍**을 보여줍니다.

결과를 보고 중복 영상과 서로 다른 영상 사이의 거리 경계를 정한 다음에만 그룹을 만듭니다.

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

VIDEO_ROOT = Path(
    '/mnt/c/ai허브/T자형교차로/095.교통사고_영상_데이터/'
    '01.데이터/1.Training/원천데이터_231108_add/'
    'TS_차대차_영상_T자형교차로'
)
REPO_ROOT = Path('/home/sungmin/blackbox-dacon')
METADATA_DIR = REPO_ROOT / 'data/stage2/aihub_tjunction/metadata'
INVENTORY_PATH = METADATA_DIR / 'inventory.csv'
FINGERPRINT_PATH = METADATA_DIR / 'brightness_fingerprints_v1.npz'
QUALITY_PATH = METADATA_DIR / 'video_quality_v1.csv'
CANDIDATE_PATH = METADATA_DIR / 'duplicate_candidates_v1.csv'

SAMPLE_RATIOS = np.array([0.10, 0.30, 0.50, 0.70, 0.90])
REBUILD_FINGERPRINTS = False

assert VIDEO_ROOT.is_dir(), VIDEO_ROOT
assert INVENTORY_PATH.is_file(), '먼저 01 노트북을 실행해 inventory.csv를 만드세요.'

inventory = pd.read_csv(INVENTORY_PATH)
inventory = inventory[inventory['read_ok']].reset_index(drop=True)
print(f'분석할 영상: {len(inventory):,}개')

## 밝기에 둔감한 영상 지문 만들기

각 프레임을 흑백으로 바꾸고 히스토그램을 평활화한 뒤 저주파 구조만 63비트로 저장합니다. 균일한 밝기 변화는 DCT의 DC 성분에 주로 나타나므로 DC 성분은 지문에서 제외합니다.

In [ ]:
def brightness_invariant_phash(frame_bgr: np.ndarray) -> np.ndarray:
    gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
    normalized = cv2.equalizeHist(gray)
    resized = cv2.resize(normalized, (32, 32), interpolation=cv2.INTER_AREA)
    dct = cv2.dct(resized.astype(np.float32))
    low_frequency = dct[:8, :8].reshape(-1)[1:]  # 전체 밝기인 DC 성분 제외
    return low_frequency > np.median(low_frequency)


def frame_quality(frame_bgr: np.ndarray) -> tuple[float, float, float, float]:
    gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
    return (
        float(gray.mean()),
        float((gray <= 5).mean()),
        float((gray >= 250).mean()),
        float(cv2.Laplacian(gray, cv2.CV_64F).var()),
    )


def fingerprint_video(path: Path, frame_count: int):
    indices = np.rint((frame_count - 1) * SAMPLE_RATIOS).astype(int)
    capture = cv2.VideoCapture(str(path))
    hashes, qualities = [], []

    for frame_index in indices:
        capture.set(cv2.CAP_PROP_POS_FRAMES, int(frame_index))
        ok, frame = capture.read()
        if not ok:
            capture.release()
            raise RuntimeError(f'프레임 읽기 실패: {path.name} @ {frame_index}')
        hashes.append(brightness_invariant_phash(frame))
        qualities.append(frame_quality(frame))

    capture.release()
    return np.stack(hashes), np.asarray(qualities, dtype=np.float32)


if FINGERPRINT_PATH.exists() and not REBUILD_FINGERPRINTS:
    # 이 노트북이 직접 저장한 신뢰할 수 있는 로컬 캐시입니다.
    # 최초 버전의 video_ids가 object 배열이라 기존 파일 로드에 allow_pickle=True가 필요합니다.
    cache = np.load(FINGERPRINT_PATH, allow_pickle=True)
    cached_video_ids = cache['video_ids'].astype(str)
    assert np.array_equal(cached_video_ids, inventory['video_id'].astype(str).to_numpy())
    fingerprints = cache['fingerprints'].astype(bool)
    qualities = cache['qualities'].astype(np.float32)
    print('기존 지문을 불러왔습니다:', FINGERPRINT_PATH)
else:
    fingerprint_rows, quality_rows = [], []
    for row in tqdm(inventory.itertuples(index=False), total=len(inventory), desc='영상 지문 생성'):
        path = VIDEO_ROOT / row.relative_path
        hashes, frame_qualities = fingerprint_video(path, int(row.frame_count))
        fingerprint_rows.append(hashes)
        quality_rows.append(frame_qualities)

    fingerprints = np.stack(fingerprint_rows)
    qualities = np.stack(quality_rows)
    np.savez_compressed(
        FINGERPRINT_PATH,
        video_ids=np.asarray(inventory['video_id'].astype(str).tolist(), dtype=np.str_),
        fingerprints=fingerprints,
        qualities=qualities,
        sample_ratios=SAMPLE_RATIOS,
    )
    print('지문 저장 완료:', FINGERPRINT_PATH)

quality_summary = pd.DataFrame({
    'video_id': inventory['video_id'],
    'brightness_mean': qualities[:, :, 0].mean(axis=1),
    'dark_clip_ratio': qualities[:, :, 1].mean(axis=1),
    'bright_clip_ratio': qualities[:, :, 2].mean(axis=1),
    'sharpness': qualities[:, :, 3].mean(axis=1),
})
quality_summary.to_csv(QUALITY_PATH, index=False, encoding='utf-8-sig')

print('지문 배열:', fingerprints.shape, '(영상, 표본 프레임, 비트)')
display(quality_summary.describe().T)

## 모든 영상에서 가장 가까운 후보 찾기

`평균 Hamming 거리`는 표본 프레임 하나당 다른 비트 수입니다. 0에 가까울수록 구조가 같습니다. 아직 임계값을 정하지 않고 거리 분포와 가장 가까운 쌍부터 확인합니다.

In [ ]:
video_count = len(fingerprints)
sample_count = fingerprints.shape[1]
distance_matrix = np.full((video_count, video_count), np.inf, dtype=np.float32)

for left_index in tqdm(range(video_count - 1), desc='영상 간 지문 비교'):
    different_bits = np.count_nonzero(
        fingerprints[left_index + 1:] != fingerprints[left_index],
        axis=(1, 2),
    )
    average_distance = different_bits / sample_count
    distance_matrix[left_index, left_index + 1:] = average_distance
    distance_matrix[left_index + 1:, left_index] = average_distance

nearest_indices = distance_matrix.argmin(axis=1)
nearest_distances = distance_matrix[np.arange(video_count), nearest_indices]
nearest = pd.DataFrame({
    'video_id': inventory['video_id'],
    'nearest_video_id': inventory.iloc[nearest_indices]['video_id'].to_numpy(),
    'nearest_hamming_per_frame': nearest_distances,
}).sort_values('nearest_hamming_per_frame')

display(nearest.head(30))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(nearest_distances, bins=50)
axes[0].set_title('각 영상의 최근접 거리 전체 분포')
axes[0].set_xlabel('평균 Hamming 거리 / 프레임')
axes[0].set_ylabel('영상 수')

axes[1].hist(nearest_distances, bins=np.arange(-0.5, 20.5, 1))
axes[1].set_xlim(-0.5, 20)
axes[1].set_title('가까운 구간 확대')
axes[1].set_xlabel('평균 Hamming 거리 / 프레임')
axes[1].set_ylabel('영상 수')
plt.tight_layout()
plt.show()

display(nearest['nearest_hamming_per_frame'].describe().to_frame('값'))

In [ ]:
left_indices, right_indices = np.triu_indices(video_count, k=1)
pair_distances = distance_matrix[left_indices, right_indices]
top_order = np.argsort(pair_distances)[:200]

top_pairs = pd.DataFrame({
    'left_index': left_indices[top_order],
    'right_index': right_indices[top_order],
    'hamming_per_frame': pair_distances[top_order],
})
top_pairs['left_video_id'] = inventory.iloc[top_pairs['left_index']]['video_id'].to_numpy()
top_pairs['right_video_id'] = inventory.iloc[top_pairs['right_index']]['video_id'].to_numpy()
top_pairs['left_brightness'] = quality_summary.iloc[top_pairs['left_index']]['brightness_mean'].to_numpy()
top_pairs['right_brightness'] = quality_summary.iloc[top_pairs['right_index']]['brightness_mean'].to_numpy()
top_pairs.to_csv(CANDIDATE_PATH, index=False, encoding='utf-8-sig')

print('중복 후보 저장:', CANDIDATE_PATH)
display(top_pairs.head(30))

## 가장 가까운 영상 쌍을 원본 밝기로 확인하기

In [ ]:
def read_frame_at_ratio(path: Path, frame_count: int, ratio: float = 0.5):
    frame_index = int(round((frame_count - 1) * ratio))
    capture = cv2.VideoCapture(str(path))
    capture.set(cv2.CAP_PROP_POS_FRAMES, frame_index)
    ok, frame = capture.read()
    capture.release()
    if not ok:
        return None
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)


pairs_to_show = top_pairs.head(12)
fig, axes = plt.subplots(len(pairs_to_show), 2, figsize=(14, 3 * len(pairs_to_show)))

for row_number, pair in enumerate(pairs_to_show.itertuples(index=False)):
    for column, video_index in enumerate([pair.left_index, pair.right_index]):
        video = inventory.iloc[video_index]
        frame = read_frame_at_ratio(
            VIDEO_ROOT / video.relative_path,
            int(video.frame_count),
        )
        if frame is not None:
            axes[row_number, column].imshow(frame)
        brightness = quality_summary.iloc[video_index]['brightness_mean']
        axes[row_number, column].set_title(
            f'{video.video_id}\n밝기 {brightness:.1f} | 거리 {pair.hamming_per_frame:.2f}',
            fontsize=9,
        )
        axes[row_number, column].axis('off')

plt.tight_layout()
plt.show()

## 거리 구간별 후보 확인하기

가장 가까운 쌍만 보면 거리 0인 사례에 치우칩니다. 아래 셀은 0, 0~1, 1~2, 2~3, 3~5, 5~8, 8~12, 12~20 구간에서 최대 두 쌍씩 뽑아 보여줍니다. 동일 영상이 마지막으로 나타나는 구간과 서로 다른 영상이 처음 나타나는 구간을 확인합니다.

In [ ]:
distance_bands = [
    ('거리 = 0', -0.001, 0.001),
    ('0 < 거리 ≤ 1', 0.001, 1.001),
    ('1 < 거리 ≤ 2', 1.001, 2.001),
    ('2 < 거리 ≤ 3', 2.001, 3.001),
    ('3 < 거리 ≤ 5', 3.001, 5.001),
    ('5 < 거리 ≤ 8', 5.001, 8.001),
    ('8 < 거리 ≤ 12', 8.001, 12.001),
    ('12 < 거리 ≤ 20', 12.001, 20.001),
    ('20 < 거리 ≤ 22', 20.001, 22.001),
    ('22 < 거리 ≤ 24', 22.001, 24.001),
    ('24 < 거리 ≤ 26', 24.001, 26.001),
    ('26 < 거리 ≤ 28', 26.001, 28.001),
    ('28 < 거리 ≤ 32', 28.001, 32.001),
    ('32 < 거리 ≤ 40', 32.001, 40.001),
]

rng = np.random.default_rng(42)
inspection_records = []
for band_name, lower, upper in distance_bands:
    positions = np.flatnonzero((pair_distances >= lower) & (pair_distances < upper))
    if len(positions) == 0:
        continue
    chosen = rng.choice(positions, size=min(2, len(positions)), replace=False)
    for position in chosen:
        inspection_records.append({
            'band': band_name,
            'left_index': int(left_indices[position]),
            'right_index': int(right_indices[position]),
            'hamming_per_frame': float(pair_distances[position]),
        })

inspection_pairs = pd.DataFrame(inspection_records)
display(inspection_pairs)

fig, axes = plt.subplots(len(inspection_pairs), 2, figsize=(14, 3 * len(inspection_pairs)))
if len(inspection_pairs) == 1:
    axes = np.asarray([axes])

for row_number, pair in enumerate(inspection_pairs.itertuples(index=False)):
    for column, video_index in enumerate([pair.left_index, pair.right_index]):
        video = inventory.iloc[video_index]
        frame = read_frame_at_ratio(
            VIDEO_ROOT / video.relative_path,
            int(video.frame_count),
        )
        if frame is not None:
            axes[row_number, column].imshow(frame)
        axes[row_number, column].set_title(
            f'{pair.band} | 거리 {pair.hamming_per_frame:.2f}\n{video.video_id}',
            fontsize=9,
        )
        axes[row_number, column].axis('off')

plt.tight_layout()
plt.show()

## 이번 노트북의 종료 지점

아직 어떠한 영상도 중복으로 확정하거나 삭제하지 않았습니다. 아래 두 결과를 확인한 후 다음 단계로 넘어갑니다.

1. `nearest_hamming_per_frame`의 요약 통계와 히스토그램
2. 위에 표시된 가장 가까운 12쌍이 실제로 밝기만 다른 동일 영상인지

이 결과로 안전한 임계값을 정한 뒤 중복 그룹과 대표 영상을 만듭니다.